# Lab Assignment 2 — Data Cleaning & Enrichment on the Heart Failure Clinical Records Dataset

**Course:** CSET343 — AI in Healthcare | B.Tech Year 4, Semester VII

**Objective:** Perform data cleaning and enrichment on a sample clinical dataset with missing values.

**Dataset:** Heart Failure Clinical Records Dataset (UCI ML Repository, id = 519), fetched directly
using `ucimlrepo`.


In [ ]:
# 1. Environment setup
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)
pd.set_option('display.max_columns', None)


## 1. Dataset Acquisition

In [ ]:
from ucimlrepo import fetch_ucirepo

heart_failure = fetch_ucirepo(id=519)

X = heart_failure.data.features
y = heart_failure.data.targets

df = pd.concat([X, y], axis=1)
print(df.shape)


## 2. Load Dataset — first 5 rows

In [ ]:
df.head()

## 3. Summarize Dataset — shape, dtypes, missing values

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
print("Column data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
df.describe().T

## 4. Visualize Missing Values — heatmap

The dataset as published has no missing values, so to practice this step we inject a small,
random amount of missingness into a working copy (`df`), using a fixed seed for reproducibility.


In [ ]:
rng = np.random.default_rng(42)
n = len(df)

def inject_missing(frame, col, frac):
    idx = rng.choice(frame.index, size=int(frac * n), replace=False)
    frame.loc[idx, col] = np.nan

inject_missing(df, 'age', 0.05)
inject_missing(df, 'ejection_fraction', 0.06)
inject_missing(df, 'serum_creatinine', 0.07)
inject_missing(df, 'platelets', 0.05)
inject_missing(df, 'smoking', 0.08)
inject_missing(df, 'sex', 0.04)

print(df.isnull().sum())


In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap (yellow = missing)')
plt.tight_layout()
plt.show()


## 5. Plot Distributions — numerical features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['age'].dropna(), kde=True, ax=axes[0], color='teal')
axes[0].set_title('Age distribution')

sns.histplot(df['ejection_fraction'].dropna(), kde=True, ax=axes[1], color='coral')
axes[1].set_title('Ejection fraction distribution')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=df['age'], ax=axes[0], color='salmon')
axes[0].set_title('Boxplot: age')

sns.boxplot(y=df['ejection_fraction'], ax=axes[1], color='salmon')
axes[1].set_title('Boxplot: ejection_fraction')

plt.tight_layout()
plt.show()


## 6. Impute Missing Values

- **Numerical** features (e.g. `serum_creatinine`): imputed with the **median**.
- **Categorical** features (e.g. `smoking`): imputed with the **mode**.


In [ ]:
num_impute_cols = ['age', 'ejection_fraction', 'serum_creatinine', 'platelets']
cat_impute_cols = ['smoking', 'sex']

for col in num_impute_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_impute_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values remaining after imputation:", df.isnull().sum().sum())


## 7. Remove Outliers

Using the IQR method on `platelets`, `serum_creatinine`, and `creatinine_phosphokinase`.


In [ ]:
def remove_outliers_iqr(frame, cols, k=1.5):
    mask = pd.Series(True, index=frame.index)
    for col in cols:
        q1, q3 = frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        mask &= frame[col].between(lower, upper)
    return frame[mask].copy()

outlier_cols = ['platelets', 'serum_creatinine', 'creatinine_phosphokinase']
rows_before = len(df)
df = remove_outliers_iqr(df, outlier_cols)
print(f"Rows before: {rows_before}, rows after: {len(df)}")


## 8. Correct Inconsistencies

Checking for physiologically invalid entries such as negative age. Since the source dataset does
not contain any, we inject a couple of synthetic invalid entries to demonstrate the check and fix.


In [ ]:
invalid_idx = rng.choice(df.dropna(subset=['age']).index, size=2, replace=False)
df.loc[invalid_idx[0], 'age'] = -5
df.loc[invalid_idx[1], 'age'] = 0

print("Invalid ages (<=0) found:", (df['age'] <= 0).sum())

df.loc[df['age'] <= 0, 'age'] = np.nan
df['age'] = df['age'].fillna(df['age'].median())

print("Invalid ages (<=0) after correction:", (df['age'] <= 0).sum())


## 9. Feature Engineering

- `age_group`: bins `<40`, `40-60`, `>60`.
- `risk_score`: normalized product of `ejection_fraction` and `serum_creatinine`.


In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, np.inf],
                          labels=['<40', '40-60', '>60'], right=False)
print(df['age_group'].value_counts())


In [ ]:
ef_norm = (df['ejection_fraction'] - df['ejection_fraction'].min()) / \
          (df['ejection_fraction'].max() - df['ejection_fraction'].min())
sc_norm = (df['serum_creatinine'] - df['serum_creatinine'].min()) / \
          (df['serum_creatinine'].max() - df['serum_creatinine'].min())

df['risk_score'] = ef_norm * sc_norm
df[['ejection_fraction', 'serum_creatinine', 'risk_score']].describe()


## 10. Encode Categorical Variables

In [ ]:
df = pd.get_dummies(df, columns=['age_group'], prefix='age_group')
print([c for c in df.columns if 'age_group' in c])

# sex and smoking are already label-encoded (0/1) in the source data
df[['sex', 'smoking']].drop_duplicates()


## 11. Normalize Features

In [ ]:
scale_cols = ['age', 'platelets', 'creatinine_phosphokinase', 'ejection_fraction',
              'serum_creatinine', 'serum_sodium', 'time']

scaler = MinMaxScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

df[scale_cols].describe().loc[['min', 'max']]


## 12. Compute Summary Statistics — before vs. after cleaning

In [ ]:
X_raw = heart_failure.data.features
summary_cols = ['age', 'ejection_fraction', 'serum_creatinine', 'platelets']

before_stats = X_raw[summary_cols].agg(['mean', 'median', 'std']).T
before_stats.columns = [f'{c}_before' for c in before_stats.columns]

after_stats = df[summary_cols].agg(['mean', 'median', 'std']).T
after_stats.columns = [f'{c}_after' for c in after_stats.columns]

pd.concat([before_stats, after_stats], axis=1)


## 13. Validate Cleaning — distributions after cleaning

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(summary_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='seagreen')
    axes[i].set_title(f'Cleaned: {col}')
plt.tight_layout()
plt.show()


## 14. Check Missing Values

In [ ]:
final_missing = df.isnull().sum()
print(final_missing)
print("Total missing values remaining:", final_missing.sum())
assert final_missing.sum() == 0
print("Confirmed: no missing values remain.")


In [ ]:
print(f"Final cleaned dataset shape: {df.shape}")
df.head()
